In [2]:
from pyesgf.search import SearchConnection

required_vars = ["tos", "sos", "hfds", "wfo", "areacello", 'msftmz']

conn = SearchConnection("https://esgf-node.llnl.gov/esg-search", distrib=True)

ctx = conn.new_context(
    project="CMIP6",
    experiment_id="piControl",
    variable_id=",".join(required_vars),   # multiple variables in one query
    facets="source_id",
    latest=True
)

models = sorted(ctx.facet_counts["source_id"].keys())

print("Models with required variables in piControl:")
for model in models:
    print(model)

Models with required variables in piControl:
ACCESS-CM2
ACCESS-ESM1-5
AWI-CM-1-1-MR
AWI-ESM-1-1-LR
BCC-CSM2-MR
BCC-ESM1
CAMS-CSM1-0
CAS-ESM2-0
CESM2
CESM2-FV2
CESM2-WACCM
CESM2-WACCM-FV2
CIESM
CMCC-CM2-SR5
CMCC-ESM2
CNRM-CM6-1
CNRM-CM6-1-HR
CNRM-ESM2-1
CanESM5
CanESM5-1
CanESM5-CanOE
E3SM-1-0
E3SM-1-1
E3SM-1-1-ECA
E3SM-2-0
E3SM-2-0-NARRM
E3SM-2-1
EC-Earth3
EC-Earth3-AerChem
EC-Earth3-CC
EC-Earth3-ESM-1
EC-Earth3-LR
EC-Earth3-Veg
EC-Earth3-Veg-LR
FGOALS-f3-L
FGOALS-g3
FIO-ESM-2-0
GFDL-CM4
GFDL-ESM4
GISS-E2-1-G
GISS-E2-1-G-CC
GISS-E2-1-H
GISS-E2-2-G
GISS-E2-2-H
GISS-E3-G
HadGEM3-GC31-LL
HadGEM3-GC31-MM
ICON-ESM-LR
IITM-ESM
INM-CM4-8
INM-CM5-0
IPSL-CM5A2-INCA
IPSL-CM6A-LR
IPSL-CM6A-MR1
KIOST-ESM
MCM-UA-1-0
MIROC-ES2H
MIROC-ES2L
MIROC6
MPI-ESM-1-2-HAM
MPI-ESM1-2-HR
MPI-ESM1-2-LR
MRI-ESM2-0
NESM3
NorCPM1
NorESM1-F
NorESM2-LM
NorESM2-MM
SAM0-UNICON
TaiESM1
UKESM1-0-LL
UKESM1-1-LL


In [3]:
from pyesgf.search import SearchConnection
import os
import requests

# =========================
# CONFIG
# =========================
models = ["MPI-ESM1-2-LR", "CESM2", "CanESM5"]
variables = ["tos", "sos", "hfds", "wfo", "msftmz"]

experiment = "piControl"
frequency = "mon"
variant = "r1i1p1f1"

save_dir = "/glade/work/stevenxu/AMOC_models/downloads/"
os.makedirs(save_dir, exist_ok=True)

# =========================
# CONNECT ESGF (distributed search)
# =========================
conn = SearchConnection(
    "https://esgf-node.llnl.gov/esg-search",
    distrib=True
)

# =========================
# FILTER: last 30 years
# =========================
def is_last_30_years(filename):
    """
    ESGF filenames look like:
    tos_Omon_MODEL_185001-201412.nc
    """
    try:
        time_range = filename.split("_")[-1].replace(".nc", "")
        start, end = time_range.split("-")
        end_year = int(end[:4])
        return end_year >= (end_year - 30)  # keeps last chunks
    except:
        return False

# =========================
# DOWNLOAD FUNCTION
# =========================
def download_file(url, filepath):
    try:
        with requests.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(filepath, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
    except Exception as e:
        print("FAILED:", filepath, e)

# =========================
# MAIN LOOP
# =========================
for model in models:
    for var in variables:

        print(f"\n=== {model} | {var} ===")

        # NOTE: use correct table for variables
        table_id = "Omon" if var != "msftmz" else "Omon"

        ctx = conn.new_context(
            project="CMIP6",
            source_id=model,
            experiment_id=experiment,
            variable=var,
            table_id=table_id,
            frequency=frequency,
            variant_label=variant,
            latest=True
        )

        results = ctx.search()

        if not results:
            print("No dataset found")
            continue

        ds = results[0]
        files = ds.file_context().search()

        print(f"Found {len(files)} files")

        for f in files:
            url = f.download_url
            filename = url.split("/")[-1]

            # 🔥 filter last 30 years
            if not is_last_30_years(filename):
                continue

            filepath = os.path.join(save_dir, filename)

            if os.path.exists(filepath):
                print("Skip:", filename)
                continue

            print("Downloading:", filename)
            download_file(url, filepath)


=== MPI-ESM1-2-LR | tos ===



-------------------------------------------------------------------------------
Warning - defaulting to search with facets=*

This behavior is kept for backward-compatibility, but ESGF indexes might not
successfully perform a distributed search when this option is used, so some
results may be missing.  For full results, it is recommended to pass a list of
facets of interest when instantiating a context object.  For example,

      ctx = conn.new_context(facets='project,experiment_id')

Only the facets that you specify will be present in the facets_counts dictionary.

This warning is displayed when a distributed search is performed while using the
facets=* default, a maximum of once per context object.  To suppress this warning,
set the environment variable ESGF_PYCLIENT_NO_FACETS_STAR_WARNING to any value
or explicitly use  conn.new_context(facets='*')

-------------------------------------------------------------------------------


HTTPError: 422 Client Error: Unprocessable Content for url: https://esgf-node.ornl.gov/esgf-1-5-bridge?format=application%2Fsolr%2Bjson&limit=0&distrib=true&type=Dataset&latest=True&project=CMIP6&source_id=MPI-ESM1-2-LR&experiment_id=piControl&variable=tos&table_id=Omon&frequency=mon&variant_label=r1i1p1f1&facets=%2A